# 02. Logistic Regression Baseline

* **Owner:** Yesid Cardenas Marin (T2)
* **Phase:** Early
* **Reads:** `load_features("fit")`, `load_features("val")`
* **Writes:**
  * `artifacts/logreg.joblib`
  * `outputs/predictions/02-lr_val.parquet`
  * `outputs/tables/02-lr_top_coefficients.csv`
  * `outputs/tables/02-lr_topk_results.csv`

---

### Purpose

Trains a baseline Logistic Regression model on the TF-IDF feature set derived from the `fit` split. Evaluates performance on the `val` split, extracts top model coefficients, computes top-$k$ metrics at hyperparameter parity, and exports all required handoff artifacts.

In [1]:
import sys
import pathlib
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Bootstrap path to access shared project module
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from shared import SEED, PATHS, load_features, compute_metrics

## 1. Feature Loading

Loads pre-transformed TF-IDF feature matrices and target labels for both `fit` and `val` splits using `shared.load_features()`.

In [2]:
X_fit, y_fit, fit_ids = load_features("fit")
X_val, y_val, val_ids = load_features("val")

print(f"Fit features shape: {X_fit.shape}")
print(f"Val features shape: {X_val.shape}")

Fit features shape: (10000, 20000)
Val features shape: (5000, 20000)


## 2. Model Training
Initializes and fits a Logistic Regression classifier on the `fit` split using a 6-point $C$ parameter grid search with 5-fold cross-validation.

In [3]:
from sklearn.model_selection import GridSearchCV

# Define a 6-point C parameter grid as budgeted in decisions.md
param_grid = {"C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}

# Set up 5-fold cross-validation grid search on fit split
grid_search = GridSearchCV(
    estimator=LogisticRegression(random_state=SEED, max_iter=1000),
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)
grid_search.fit(X_fit, y_fit)

# Select best model found by CV
model = grid_search.best_estimator_
best_c = grid_search.best_params_["C"]

print(f"Best C parameter found: {best_c}")
print(f"Best CV ROC-AUC score: {grid_search.best_score_:.4f}")

Best C parameter found: 10.0
Best CV ROC-AUC score: 0.9536


## 3. Validation & Predictions
Generates probability predictions on the `val` split, formats predictions into the standard project schema, and computes evaluation metrics.

In [4]:
# Predict probabilities and class labels on validation set
y_proba_pos = model.predict_proba(X_val)[:, 1]
y_pred = (y_proba_pos >= 0.5).astype(int)

# Create standardized validation predictions DataFrame using val_ids returned from load_features
df_lr_val = pd.DataFrame({
    "id": val_ids,
    "y_true": y_val,
    "y_pred": y_pred,
    "y_proba_pos": y_proba_pos
})

# Compute metrics
metrics = compute_metrics(y_val, y_pred, y_proba_pos)
print("Validation Metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Validation Metrics:
  accuracy: 0.8914
  precision: 0.8851
  recall: 0.8996
  f1: 0.8923
  roc_auc: 0.9581
  confusion_matrix: [[2208, 292], [251, 2249]]


## 4. Feature Coefficients & Top-K Analysis
Extracts model coefficients associated with TF-IDF features and evaluates feature selection ablation across top-$k$ feature subsets ($k \in [50, 100, 500]$) at hyperparameter parity ($C=\text{best\_c}$).

In [5]:
# Load vectorizer using the exact key defined in shared.py
vectorizer = joblib.load(PATHS["tfidf"])
feature_names = vectorizer.get_feature_names_out()

# Extract model coefficients from tuned model
coefs = model.coef_[0]
df_coefs_full = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefs
}).sort_values(by="coefficient", ascending=False)

# Take head/tail slice of top 50 positive and top 50 negative coefficients
df_coefs = pd.concat([df_coefs_full.head(50), df_coefs_full.tail(50)]).reset_index(drop=True)

# Top-K Feature Ablation Experiment (Retrain LR on top K features by absolute magnitude)
top_idx = np.argsort(np.abs(coefs))[::-1]
topk_records = []

# Try TOP_K_VALUES from shared.py, default to [50, 100, 500] if not present
try:
    from shared import TOP_K_VALUES
except ImportError:
    TOP_K_VALUES = [50, 100, 500]

for k in TOP_K_VALUES:
    cols = top_idx[:k]
    # Retrain model on only top-k features using optimal C hyperparameter
    m_k = LogisticRegression(random_state=SEED, max_iter=1000, C=best_c).fit(X_fit[:, cols], y_fit)
    p_k = m_k.predict_proba(X_val[:, cols])[:, 1]
    y_pred_k = (p_k >= 0.5).astype(int)
    
    # Compute metrics for top-k model
    k_metrics = compute_metrics(y_val, y_pred_k, p_k)
    k_metrics["k"] = k
    topk_records.append(k_metrics)

df_topk = pd.DataFrame(topk_records)
print("Top-K Feature Ablation Results:")
print(df_topk[["k", "accuracy", "f1", "roc_auc"]])

Top-K Feature Ablation Results:
     k  accuracy        f1   roc_auc
0   50    0.8164  0.824003  0.893494
1  100    0.8376  0.842391  0.915702
2  500    0.8748  0.876723  0.946148


## 5. Export Handoff Artifacts
Saves trained model objects, predictions, coefficient tables, and top-$k$ results to disk in accordance with the project contract.

In [6]:
# Ensure destination directories exist
PATHS["artifacts_dir"].mkdir(parents=True, exist_ok=True)
PATHS["predictions_dir"].mkdir(parents=True, exist_ok=True)
PATHS["tables_dir"].mkdir(parents=True, exist_ok=True)

# 1. Save trained model artifact (unprefixed shared pipeline artifact)
joblib.dump(model, PATHS["artifacts_dir"] / "logreg.joblib")

# 2. Save validation predictions (notebook output with '02-' prefix)
df_lr_val.to_parquet(PATHS["predictions_dir"] / "02-lr_val.parquet", index=False)

# 3. Save sliced coefficient table (notebook output with '02-' prefix)
df_coefs.to_csv(PATHS["tables_dir"] / "02-lr_top_coefficients.csv", index=False)

# 4. Save top-k feature ablation results (notebook output with '02-' prefix)
df_topk.to_csv(PATHS["tables_dir"] / "02-lr_topk_results.csv", index=False)

print("Handoff artifacts successfully saved.")

Handoff artifacts successfully saved.
